# 45 — Avaliação OOD: 4 corpora externos × todos os modelos

Aplica os modelos treinados (BERT, TF-IDF — binário e multiclasse) sobre quatro corpora externos.

| Dataset | Positivo binário | Níveis avaliados |
|---|---|---|
| **Fake.Br** (Lira et al.) | `metadata_category == 'economia'` (44/7200, 0.6%) | 1 binário, 2 multiclasse, 3 subgrupo por veracidade |
| **FakeRecogna** | URL do portal `economia.uol.com.br` (312/11872, 2.6%) | 1 binário, 2 multiclasse, 3 corte balanceado intra-UOL |
| **PortugueseNewsDataset** (Klaifer/WikiNotícias, PLOS ONE 2024) | `category == 'Economia e negócios'` (1.151/9.135, 12.6%) | 1 binário (partição `full`) |
| **RecognaSumm** (Paiola et al., PROPOR 2024) | `Categoria == 'Economia'` (12.613/135.272, 9.3%) | 1 binário (partição `full` — train+validation+test) |

Cada nível emite `result_card.json` em `<DRIVE>/ood_runs/<model_id>_<task>_<dataset>_<level>/`. A célula final agrega cards e roda McNemar pareado por `(domain, level)`.

**Escopo binário-only de PN/RS**: PortugueseNewsDataset e RecognaSumm são avaliados **somente no nível binário** (label de tópico direta, sem mapeamento multiclasse). Os modelos multiclasse rodam apenas em Fake.Br e FakeRecogna.

**Pré-requisitos**:
- `colab_ood_data.zip` (zip unificado dos 4 corpora) em `<DRIVE>/economy-classifier/`. Gerar localmente com `uv run python scripts/colab_pack_ood_data.py` (vide seção 3). Para `RS_PARTITION='full'` o pack inclui `train.jsonl`+`validation.jsonl`+`test.jsonl` (~460 MB descompactados).
- Modelos treinados em `<DRIVE>/economy-classifier/runs/<model_id>_<task>_test_set/model/` (HF dir para BERT, `tfidf_pipeline.joblib` para TF-IDF), gerados pelos NBs 21 / 11 / 12 / 13.

## 0. Verificação de ambiente

In [ ]:
import torch

if not torch.cuda.is_available():
    print("AVISO: GPU nao detectada. BERT vai rodar em CPU (lento). "
          "Para acelerar: Runtime > Change runtime type > GPU.")
    GPU_NAME = "CPU"
    HARDWARE = "Colab-CPU"
else:
    GPU_NAME = torch.cuda.get_device_name(0)
    VRAM_GB = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    print(f"GPU: {GPU_NAME} ({VRAM_GB} GB VRAM)")
    HARDWARE = f"Colab-{GPU_NAME.split()[-1]}"
print("CUDA:", torch.version.cuda)


AVISO: GPU nao detectada. BERT vai rodar em CPU (lento). Para acelerar: Runtime > Change runtime type > GPU.
CUDA: None


## 1. Bootstrap (Colab + local)

In [ ]:
import subprocess
import sys
import zipfile
from pathlib import Path


def _run(cmd: list[str], description: str) -> None:
    print(f"$ {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr, file=sys.stderr)
        raise RuntimeError(f"{description} failed with exit code {result.returncode}")


IN_COLAB = "google.colab" in sys.modules
print("Ambiente:", "Google Colab" if IN_COLAB else "Local")
print("Python   :", sys.version.split()[0])

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_URL = "https://github.com/almeidadm/economy-classifier.git"
    REPO_BRANCH = "main"
    DRIVE_FOLDER = "economy-classifier"

    DRIVE_BASE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    REPO_DIR = Path("/content/economy-classifier")

    if REPO_DIR.exists():
        _run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], "git fetch")
        _run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], "git checkout")
        _run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"], "git reset")
    else:
        _run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], "git clone")

    _run(
        [sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR),
         "--upgrade-strategy", "only-if-needed", "-q"],
        "pip install -e .",
    )

    if str(REPO_DIR / "src") not in sys.path:
        sys.path.insert(0, str(REPO_DIR / "src"))

    RUNS_BASE = DRIVE_BASE / "runs"
    OOD_RUNS_BASE = DRIVE_BASE / "ood_runs"
    OOD_DATA_DIR = Path("/content/ood_data")
else:
    REPO_DIR = Path.cwd().parent
    DRIVE_BASE = REPO_DIR / "artifacts"
    RUNS_BASE = DRIVE_BASE / "runs"
    OOD_RUNS_BASE = DRIVE_BASE / "ood_runs"
    OOD_DATA_DIR = REPO_DIR / "ood_data"

OOD_RUNS_BASE.mkdir(parents=True, exist_ok=True)
OOD_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_DIR     :", REPO_DIR)
print("RUNS_BASE    :", RUNS_BASE)
print("OOD_RUNS_BASE:", OOD_RUNS_BASE)
print("OOD_DATA_DIR :", OOD_DATA_DIR)


Ambiente: Google Colab
Python   : 3.12.13
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
$ git -C /content/economy-classifier fetch origin main
$ git -C /content/economy-classifier checkout main
Your branch is up to date with 'origin/main'.

$ git -C /content/economy-classifier reset --hard origin/main
HEAD is now at a602898 feat(ood-eval): scripts + notebook para avaliacao OOD em Fake.Br + FakeRecogna

$ /usr/bin/python3 -m pip install -e /content/economy-classifier --upgrade-strategy only-if-needed -q
REPO_DIR     : /content/economy-classifier
RUNS_BASE    : /content/drive/MyDrive/economy-classifier/runs
OOD_RUNS_BASE: /content/drive/MyDrive/economy-classifier/ood_runs
OOD_DATA_DIR : /content/ood_data


## 2. Imports dos scripts de avaliação

In [ ]:
# scripts/ nao e um pacote Python — adicionamos manualmente ao path
SCRIPTS_DIR = REPO_DIR / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import evaluate_fake_br as fb
import evaluate_fake_recogna as fr
import evaluate_portuguese_news as pn
import evaluate_recognasumm as rs
from economy_classifier.project import compute_artifact_size_mb
from economy_classifier.evaluation import compute_mcnemar_pairwise

FAKE_BR_ROOT = OOD_DATA_DIR / "fake_br"
FAKE_RECOGNA_ROOT = OOD_DATA_DIR / "fake_recogna"
PORTUGUESE_NEWS_ROOT = OOD_DATA_DIR / "portuguese_news_wikinotices"
RECOGNASUMM_ROOT = OOD_DATA_DIR / "recognasumm"

print("FAKE_BR_ROOT        :", FAKE_BR_ROOT)
print("FAKE_RECOGNA_ROOT   :", FAKE_RECOGNA_ROOT)
print("PORTUGUESE_NEWS_ROOT:", PORTUGUESE_NEWS_ROOT)
print("RECOGNASUMM_ROOT    :", RECOGNASUMM_ROOT)

## 3. Carregamento dos datasets OOD

**Layout esperado** dentro de `colab_ood_data.zip` (zip unificado para os 4 corpora):

```
fake_br/
  full_texts/
    fake/<id>.txt
    true/<id>.txt
    fake-meta-information/<id>-meta.txt
    true-meta-information/<id>-meta.txt
fake_recogna/
  FakeRecogna_*.xlsx
portuguese_news_wikinotices/
  wikinews_categories.json
  wikinews_train.json
  wikinews_test.json
  split_ids.csv
recognasumm/
  train.jsonl
  validation.jsonl
  test.jsonl
```

**Para gerar o zip localmente** (uma vez, antes do primeiro upload):

```bash
cd ~/Documentos/repositorios/economy-classifier
uv run python scripts/colab_pack_ood_data.py
# Sobe colab_ood_data.zip para <DRIVE>/economy-classifier/
```

O script valida que:
- `fn-dataset-eda/data/raw/{fake_br,fake_recogna}/` existem (sibling repo).
- `data/portuguese_news_wikinotices/{wikinews_*.json,split_ids.csv}` existem (reconstruir via repo Klaifer — ver `docs/colab_run_portuguese_news.md`).
- `data/recognasumm/{train,validation,test}.jsonl` existem (baixar do HF — ver `docs/colab_run_recognasumm.md`).

Em execução **local**, a célula abaixo cria links simbólicos diretos para essas fontes (sem precisar empacotar/desempacotar).

In [ ]:
PN_PARTITION = "full"   # 9135 docs; troque para "test" (914) para reproduzir paper Klaifer
RS_PARTITION = "full"   # 135.272 docs (train+validation+test); requer 3 jsonl no zip


if IN_COLAB:
    zip_path = DRIVE_BASE / "colab_ood_data.zip"
    expected_roots = [FAKE_BR_ROOT, FAKE_RECOGNA_ROOT, PORTUGUESE_NEWS_ROOT, RECOGNASUMM_ROOT]
    if not all(r.exists() for r in expected_roots):
        assert zip_path.exists(), (
            f"Falta {zip_path}. Rode scripts/colab_pack_ood_data.py local e suba para o Drive."
        )
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(OOD_DATA_DIR)
        print(f"Extraido {zip_path.name} -> {OOD_DATA_DIR}")
else:
    LOCAL_FN_REPO = Path("/home/diacrono/Documentos/repositorios/fn-dataset-eda/data/raw")
    if not FAKE_BR_ROOT.exists() and (LOCAL_FN_REPO / "fake_br").exists():
        FAKE_BR_ROOT.symlink_to(LOCAL_FN_REPO / "fake_br")
    if not FAKE_RECOGNA_ROOT.exists() and (LOCAL_FN_REPO / "fake_recogna").exists():
        FAKE_RECOGNA_ROOT.symlink_to(LOCAL_FN_REPO / "fake_recogna")
    LOCAL_PN = REPO_DIR / "data" / "portuguese_news_wikinotices"
    LOCAL_RS = REPO_DIR / "data" / "recognasumm"
    if not PORTUGUESE_NEWS_ROOT.exists() and LOCAL_PN.exists():
        PORTUGUESE_NEWS_ROOT.symlink_to(LOCAL_PN)
    if not RECOGNASUMM_ROOT.exists() and LOCAL_RS.exists():
        RECOGNASUMM_ROOT.symlink_to(LOCAL_RS)

assert FAKE_BR_ROOT.exists(), f"Fake.Br ausente em {FAKE_BR_ROOT}"
assert FAKE_RECOGNA_ROOT.exists(), f"FakeRecogna ausente em {FAKE_RECOGNA_ROOT}"
assert PORTUGUESE_NEWS_ROOT.exists(), f"PortugueseNewsDataset ausente em {PORTUGUESE_NEWS_ROOT}"
assert RECOGNASUMM_ROOT.exists(), f"RecognaSumm ausente em {RECOGNASUMM_ROOT}"

print("Carregando Fake.Br...")
fb_df = fb.load_fake_br(FAKE_BR_ROOT)
print(f"  {len(fb_df)} docs | economia={int((fb_df.y_true_binary==1).sum())}")

print("Carregando FakeRecogna...")
fr_df = fr.load_fake_recogna(FAKE_RECOGNA_ROOT)
print(f"  {len(fr_df)} docs | economia.uol={int((fr_df.y_true_binary==1).sum())}")

print(f"Carregando PortugueseNewsDataset (partition={PN_PARTITION})...")
pn_df = pn.load_portuguese_news(PORTUGUESE_NEWS_ROOT, partition=PN_PARTITION)
print(f"  {len(pn_df)} docs | Economia e negocios={int((pn_df.y_true_binary==1).sum())}")

print(f"Carregando RecognaSumm (partition={RS_PARTITION})...")
rs_df = rs.load_recognasumm(RECOGNASUMM_ROOT, partition=RS_PARTITION)
print(f"  {len(rs_df)} docs | Economia={int((rs_df.y_true_binary==1).sum())}")

# Caches reaproveitados em todos os modelos
fb_texts = fb_df["text"].fillna("").tolist()
fr_texts = fr_df["text"].fillna("").tolist()
pn_texts = pn_df["text"].fillna("").tolist()
rs_texts = rs_df["text"].fillna("").tolist()

## 3.5 Caracterização do shift representacional (MMD² / KTS)

Quantifica a distância entre o pool de treino do FolhaSP e cada corpus OOD em espaço de embeddings, **com encoder fixo**. Permite ler a queda de F1 em §6 como função da magnitude do shift, em vez de número solto.

**Encoder:** `neuralmind/bert-base-portuguese-cased` (BERTimbau base, **sem fine-tune**). Razão: queremos medir deriva em $P(x)$ — usar um classificador fine-tuned misturaria deriva de corpus com viés do próprio encoder treinado.

**Subamostragem:** $n=1500$ por lado, $R=5$ replicatas independentes. Matriz de kernel cabe em ~36 MB; permutação reusa a matriz pré-computada.

**Caminho do pool de referência:** `artifacts/splits/train.parquet` (extraído de `colab_splits.zip` no Colab, igual aos NBs 11/12/13/21).

In [ ]:
import time

import numpy as np
import pandas as pd
from transformers import AutoModel, AutoTokenizer

ENCODER_ID = "neuralmind/bert-base-portuguese-cased"
EMB_MAX_LEN = 128
EMB_BATCH = 32
CAP_PER_CORPUS = 8000
N_SUBSAMPLE = 1500
N_REPLICATES = 5
N_PERMUTATIONS = 500
SHIFT_SEED = 2026

emb_device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device para embeddings:", emb_device)

# Pool de referencia FolhaSP: train do 90/10 — mesma fonte que alimentou os modelos.
SPLITS_DIR = REPO_DIR / "artifacts" / "splits"
if IN_COLAB and not (SPLITS_DIR / "train.parquet").exists():
    splits_zip = DRIVE_BASE / "colab_splits.zip"
    assert splits_zip.exists(), f"Falta {splits_zip} no Drive."
    with zipfile.ZipFile(splits_zip) as zf:
        zf.extractall(REPO_DIR / "artifacts")
    print(f"Extraido {splits_zip.name}")
assert (SPLITS_DIR / "train.parquet").exists(), f"Falta {SPLITS_DIR/'train.parquet'}"

folha_train = pd.read_parquet(SPLITS_DIR / "train.parquet")
folha_texts_all = folha_train["text"].fillna("").tolist()
print(f"FolhaSP train pool: {len(folha_texts_all)} docs")

tokenizer = AutoTokenizer.from_pretrained(ENCODER_ID)
encoder = AutoModel.from_pretrained(ENCODER_ID).to(emb_device).eval()


@torch.no_grad()
def embed_texts(texts, batch_size=EMB_BATCH, max_length=EMB_MAX_LEN):
    """Mean-pooled embeddings (last hidden state) com mascara de atencao."""
    out = []
    for i in range(0, len(texts), batch_size):
        chunk = [t if t else "[PAD]" for t in texts[i:i+batch_size]]
        enc = tokenizer(chunk, padding=True, truncation=True,
                        max_length=max_length, return_tensors="pt").to(emb_device)
        h = encoder(**enc).last_hidden_state
        mask = enc["attention_mask"].unsqueeze(-1).float()
        pooled = (h * mask).sum(1) / mask.sum(1).clamp(min=1)
        out.append(pooled.cpu().numpy())
    return np.vstack(out)


shift_rng = np.random.default_rng(SHIFT_SEED)


def sample_and_embed(texts, name, cap=CAP_PER_CORPUS):
    n = min(len(texts), cap)
    idx = shift_rng.choice(len(texts), size=n, replace=False)
    chosen = [texts[int(i)] for i in idx]
    t0 = time.perf_counter()
    emb = embed_texts(chosen)
    print(f"  {name:<15s} {emb.shape}  {time.perf_counter()-t0:.1f}s")
    return emb


print("\nGerando embeddings (encoder fixo, sem fine-tune):")
emb_folha = sample_and_embed(folha_texts_all, "FolhaSP")
emb_ood = {
    "fake_br":                     sample_and_embed(fb_texts, "Fake.Br"),
    "fake_recogna":                sample_and_embed(fr_texts, "FakeRecogna"),
    "portuguese_news_wikinotices": sample_and_embed(pn_texts, "PortugueseNews"),
    "recognasumm_propor2024":      sample_and_embed(rs_texts, "RecognaSumm"),
}


## 3.6 MMD² e KTS via permutação

Estimador biased de MMD² com kernel RBF; largura $\sigma$ pela heurística da mediana sobre $X \cup Y$. p-valor por permutação ($B = N\_PERMUTATIONS$): embaralha rótulos de origem e recomputa MMD² via reindexação da matriz de kernel pré-computada — evita reembedar a cada permutação.

Reporta `mmd2_mean ± std`, `p_median` e `n_reject` (replicatas com $p<0.05$) por OOD.

In [ ]:
from sklearn.metrics.pairwise import rbf_kernel
from scipy.spatial.distance import pdist


def median_heuristic_sigma(Z, rng, n_probe=1000):
    sub = Z[rng.choice(len(Z), size=min(n_probe, len(Z)), replace=False)]
    d = pdist(sub)
    return float(np.sqrt(np.median(d**2) / 2))


def mmd2_biased(K, n):
    """Estimador biased de MMD^2 a partir da matriz de kernel ja calculada.

    K e o kernel do bloco [X; Y]; n=|X|; m=|Y|=K.shape[0]-n.
    """
    m = K.shape[0] - n
    K_xx = K[:n, :n].sum() / (n * n)
    K_yy = K[n:, n:].sum() / (m * m)
    K_xy = K[:n, n:].sum() / (n * m)
    return float(K_xx + K_yy - 2 * K_xy)


def kts_permutation(X, Y, n_perm, rng):
    """KTS = teste de hipotese baseado em MMD^2 com permutacao.

    Retorna (mmd2_observado, p_valor, sigma_usado). p e a fracao das
    permutacoes em que MMD^2 permutado >= observado.
    """
    Z = np.vstack([X, Y])
    n = len(X)
    sigma = median_heuristic_sigma(Z, rng)
    K = rbf_kernel(Z, gamma=1.0 / (2 * sigma ** 2))
    observed = mmd2_biased(K, n)
    idx_base = np.arange(len(Z))
    null = np.empty(n_perm)
    for b in range(n_perm):
        perm = rng.permutation(idx_base)
        Kp = K[np.ix_(perm, perm)]
        null[b] = mmd2_biased(Kp, n)
    p = float((null >= observed).mean())
    return observed, p, sigma


shift_rows = []
for ood_name, emb_Y in emb_ood.items():
    n_per_side = min(N_SUBSAMPLE, len(emb_folha), len(emb_Y))
    print(f"\n{ood_name}: n={n_per_side} por lado, {N_REPLICATES} replicatas")
    for r in range(N_REPLICATES):
        ix = shift_rng.choice(len(emb_folha), size=n_per_side, replace=False)
        iy = shift_rng.choice(len(emb_Y),     size=n_per_side, replace=False)
        mmd2, p, sigma = kts_permutation(
            emb_folha[ix], emb_Y[iy],
            n_perm=N_PERMUTATIONS, rng=shift_rng,
        )
        shift_rows.append({
            "ood": ood_name, "replicate": r, "n_per_side": n_per_side,
            "mmd2": mmd2, "p_value": p, "sigma": sigma,
        })
        print(f"  rep {r}: MMD2={mmd2:.4f}  p={p:.4f}  sigma={sigma:.3f}")

shift_df = pd.DataFrame(shift_rows)
shift_summary = (
    shift_df.groupby("ood")
    .agg(
        mmd2_mean=("mmd2", "mean"),
        mmd2_std=("mmd2", "std"),
        p_median=("p_value", "median"),
        n_reject=("p_value", lambda s: int((s < 0.05).sum())),
    )
    .round(4)
)
display(shift_summary)


## 3.7 Baseline de calibração: FolhaSP × FolhaSP

Sem referência interna, a magnitude de MMD² não é interpretável. Duas subamostras disjuntas do próprio FolhaSP devem produzir MMD² ≈ 0 e $p > 0.05$ — é o "ruído amostral" da pipeline e a linha de base contra a qual os MMD² OOD são lidos.

In [ ]:
half = len(emb_folha) // 2
n_self = min(N_SUBSAMPLE, half)
self_rows = []
print(f"Baseline FolhaSP-vs-FolhaSP: n={n_self} por lado, {N_REPLICATES} replicatas")
for r in range(N_REPLICATES):
    ix_a = shift_rng.choice(half, size=n_self, replace=False)
    ix_b = half + shift_rng.choice(len(emb_folha) - half, size=n_self, replace=False)
    mmd2, p, sigma = kts_permutation(emb_folha[ix_a], emb_folha[ix_b],
                                     n_perm=N_PERMUTATIONS, rng=shift_rng)
    self_rows.append({"replicate": r, "mmd2": mmd2, "p_value": p, "sigma": sigma})
    print(f"  rep {r}: MMD2={mmd2:.4f}  p={p:.4f}")

self_df = pd.DataFrame(self_rows)
shift_summary.loc["folha_self_control"] = {
    "mmd2_mean": round(self_df["mmd2"].mean(), 4),
    "mmd2_std":  round(self_df["mmd2"].std(),  4),
    "p_median":  round(self_df["p_value"].median(), 4),
    "n_reject":  int((self_df["p_value"] < 0.05).sum()),
}
display(shift_summary)


## 3.8 Distâncias EMD / FTD² / MAUVE (framework `metric-viz`)

Complementa MMD²/KTS (§3.5–3.7) com as três métricas de dissimilaridade entre
distribuições textuais do repositório vizinho [`metric-viz`](../metric-viz):

| Métrica | O que mede | Direção |
|---|---|---|
| **EMD** (Wasserstein) | custo de transporte ótimo entre as nuvens de embeddings | ↑ = mais distante |
| **FTD²** (Fréchet Text Distance) | distância gaussiana (média + covariância), análogo ao FID | ↑ = mais distante |
| **MAUVE** | sobreposição via fronteira de divergência KL (clusters _k_-means) | ↓ = mais distante |

Reusa os **mesmos embeddings de §3.5** (`emb_folha`, `emb_ood` — BERTimbau base sem
fine-tune, `max_length=128`), agora **L2-normalizados** (semântica idêntica ao
`metric-viz`). Subamostra `N_DIST=200` por lado com `DIST_REPLICATES` replicatas; o EMD
é transporte ótimo **exato** (`linprog/highs`), `O(n²)` variáveis, daí `n` modesto.

> ⚠️ **Confound de comprimento (declarar no artigo).** EMD e FTD² são fortemente
> sensíveis ao comprimento do texto: no `metric-viz`, **~97% do EMD e ~93% do FTD²**
> da comparação FolhaUOL × FakeRecogna são **puro artefato de comprimento**, não
> deslocamento de domínio (ver §3.9). Apenas **MAUVE é robusto a comprimento**. Aqui a
> truncagem comum (`max_length=128`) atenua — mas **não elimina** — o efeito para corpora
> de granularidade muito diferente (FakeRecogna é manchete; os demais são corpo). Trate
> EMD/FTD² como descritivos; **MAUVE** é a leitura defensável de deslocamento de $P(x)$.

> **Referência ≠ `metric-viz`.** Aqui a referência é o **pool de treino FolhaSP**
> (`artifacts/splits/train.parquet`, todas as categorias — o que os modelos viram),
> coerente com a leitura OOD deste NB. O `metric-viz` fixa **FolhaUOL só-`mercado`**;
> por isso os valores absolutos diferem (a §3.9 traz os números do `metric-viz` como
> documentados).

In [ ]:
from scipy.optimize import linprog
from scipy import sparse
from scipy.linalg import sqrtm
from scipy.spatial.distance import cdist
from sklearn.covariance import LedoitWolf
from sklearn.cluster import KMeans
from sklearn.metrics import auc
from sklearn.preprocessing import normalize

N_DIST = 200            # docs por lado; EMD exato e O(n^2) variaveis -> n modesto
DIST_REPLICATES = 3     # replicatas independentes (subamostragem)


# --- Metricas verbatim de metric-viz/notebooks/build_vs_todos.py ---
def emd_ot(X, Y):
    n, m = len(X), len(Y)
    C = cdist(X, Y)
    A_eq = sparse.vstack([sparse.kron(sparse.eye(n), np.ones((1, m))),
                          sparse.kron(np.ones((1, n)), sparse.eye(m))])
    b_eq = np.concatenate([np.full(n, 1 / n), np.full(m, 1 / m)])
    res = linprog(C.ravel(), A_eq=A_eq, b_eq=b_eq, bounds=(0, None), method="highs")
    return float(res.fun)


def ftd2(X, Y):
    Sx = LedoitWolf().fit(X).covariance_
    Sy = LedoitWolf().fit(Y).covariance_
    cm = sqrtm(Sx @ Sy)
    cm = cm.real if np.iscomplexobj(cm) else cm
    return float(np.sum((X.mean(0) - Y.mean(0)) ** 2) + np.trace(Sx + Sy - 2 * cm))


def mauve_score(X, Y, k=10, c=5.0, n_lambda=80, seed=SHIFT_SEED):
    Z = np.vstack([X, Y])
    lab = KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(Z)
    P = np.bincount(lab[:len(X)], minlength=k).astype(float); P /= P.sum()
    Q = np.bincount(lab[len(X):], minlength=k).astype(float); Q /= Q.sum()
    eps = 1e-10
    KL = lambda a, b: np.sum(np.where(a > 0, a * np.log((a + eps) / (b + eps)), 0))
    curva = [[0.0, np.inf]]
    for lam in np.linspace(1e-3, 1 - 1e-3, n_lambda):
        R = lam * P + (1 - lam) * Q
        curva.append([KL(Q, R), KL(P, R)])
    curva.append([np.inf, 0.0])
    curva = np.exp(-c * np.asarray(curva)); xs, ys = curva[:, 0], curva[:, 1]
    o1, o2 = np.argsort(xs), np.argsort(ys)
    return float(0.5 * (auc(xs[o1], ys[o1]) + auc(ys[o2], xs[o2])))


# L2-normaliza copias dos embeddings de 3.5 (semantica do metric-viz)
emb_folha_n = normalize(emb_folha)
emb_ood_n = {k: normalize(v) for k, v in emb_ood.items()}

dist_rng = np.random.default_rng(SHIFT_SEED)
dist_rows = []

# baseline FolhaSP<->FolhaSP (metades disjuntas) -- piso "sem shift"
half = len(emb_folha_n) // 2
for r in range(DIST_REPLICATES):
    ia = dist_rng.choice(half, size=min(N_DIST, half), replace=False)
    ib = half + dist_rng.choice(len(emb_folha_n) - half,
                                size=min(N_DIST, len(emb_folha_n) - half), replace=False)
    Xa, Yb = emb_folha_n[ia], emb_folha_n[ib]
    dist_rows.append({"ood": "folha_self_control", "replicate": r,
                      "EMD": emd_ot(Xa, Yb), "FTD2": ftd2(Xa, Yb), "MAUVE": mauve_score(Xa, Yb)})
print(f"baseline folha_self_control: {DIST_REPLICATES} replicatas OK")

for ood_name, emb_Y in emb_ood_n.items():
    n_side = min(N_DIST, len(emb_folha_n), len(emb_Y))
    print(f"{ood_name}: n={n_side} por lado, {DIST_REPLICATES} replicatas")
    for r in range(DIST_REPLICATES):
        ia = dist_rng.choice(len(emb_folha_n), size=n_side, replace=False)
        iy = dist_rng.choice(len(emb_Y),       size=n_side, replace=False)
        Xa, Yb = emb_folha_n[ia], emb_Y[iy]
        dist_rows.append({"ood": ood_name, "replicate": r,
                          "EMD": emd_ot(Xa, Yb), "FTD2": ftd2(Xa, Yb), "MAUVE": mauve_score(Xa, Yb)})

dist_df = pd.DataFrame(dist_rows)
dist_summary = (
    dist_df.groupby("ood")
    .agg(EMD_mean=("EMD", "mean"), EMD_std=("EMD", "std"),
         FTD2_mean=("FTD2", "mean"), FTD2_std=("FTD2", "std"),
         MAUVE_mean=("MAUVE", "mean"), MAUVE_std=("MAUVE", "std"))
    .round(4)
)
# junta MMD2 (3.5) na mesma tabela quando disponivel
if "shift_summary" in globals():
    dist_summary = dist_summary.join(shift_summary[["mmd2_mean"]], how="left")
print("\nDistancias (referencia = pool de treino FolhaSP; EMD/FTD2^ MAUVEv):")
display(dist_summary)

## 3.9 Decomposição comprimento × domínio (FakeRecogna) — resultado do `metric-viz`

A única variante de FakeRecogna com texto *raw* é a **manchete** (`Título + Subtítulo`,
~75 chars) — a coluna `Notícia` é lematizada/lowercased e imprópria para embedding
semântico. Os demais corpora usam corpo (~2000 chars). Logo, uma distância
FolhaUOL(corpo) × FakeRecogna(manchete) **mistura deslocamento de domínio com
deslocamento de comprimento/granularidade**.

O script [`compute_fakerecogna_distance.py`](../metric-viz/compute_fakerecogna_distance.py)
isola os dois efeitos com BERTimbau (mean-pool L2, `max_length=256`, `seed=42`,
FolhaUOL `mercado` × FakeRecogna `economia.uol.com.br`, $n=200$/lado):

- **baseline corpo**: FolhaUOL(corpo) ↔ FolhaUOL(corpo) — piso sem shift
- **baseline título**: FolhaUOL(título) ↔ FolhaUOL(título) — piso só-manchete
- **matched**: FolhaUOL(título) ↔ FakeRecogna(tít+sub) — granularidade casada (domínio "limpo")
- **body_ref**: FolhaUOL(tít+corpo) ↔ FakeRecogna(tít+sub) — comparável à tabela atual, *length-confounded*

**Moral**: ~**97% do EMD** e ~**93% do FTD²** entre FolhaUOL e FakeRecogna são
artefato de comprimento; só **MAUVE** mede deslocamento de domínio de forma robusta
(quase imune ao comprimento). A célula abaixo reproduz a tabela e a decomposição
**já calculadas** no `metric-viz` (constantes documentadas — a fonte FolhaUOL + manchete
raw não está no layout de dados deste NB).

In [ ]:
# Resultado PRE-CALCULADO em ../metric-viz/compute_fakerecogna_distance.py
# (BERTimbau mean-pool L2, max_length=256, seed=42, n=200/lado). Inline como
# constante documentada: a fonte (FolhaUOL/mercado + manchete raw do FakeRecogna)
# nao esta no layout de dados deste NB. Regenerar: `python compute_fakerecogna_distance.py`.
fakerecogna_distance = {
    "reference": "FolhaUOL category=='mercado'",
    "target": "FakeRecogna netloc=='economia.uol.com.br' (positivos OOD)",
    "embedding": "bertimbau mean-pool L2, max_length=256",
    "n_reference": 200, "n_target_sampled": 200, "n_target_available": 312, "seed": 42,
    "rows": [
        {"comparacao": "folhauol[corpo]<->folhauol[corpo] (baseline corpo)",   "EMD": 0.3911, "FTD2": 0.0445, "MAUVE": 0.9573},
        {"comparacao": "folhauol[titulo]<->folhauol[titulo] (baseline titulo)", "EMD": 0.6890, "FTD2": 0.1200, "MAUVE": 0.9477},
        {"comparacao": "folhauol[titulo]<->fakerecogna[tit+sub] (matched)",     "EMD": 0.6975, "FTD2": 0.1254, "MAUVE": 0.7798},
        {"comparacao": "folhauol[tit+corpo]<->fakerecogna[tit+sub] (body_ref)", "EMD": 0.7903, "FTD2": 0.4253, "MAUVE": 0.0040},
    ],
    "decomposition": {
        "EMD":  {"length_artifact": 0.2979, "domain_shift": 0.0085, "total": 0.3064, "pct_artifact": 97.2},
        "FTD2": {"length_artifact": 0.0755, "domain_shift": 0.0054, "total": 0.0809, "pct_artifact": 93.3},
        "MAUVE": {"domain_shift_drop": 0.1679, "note": "MAUVE ~imune ao comprimento"},
    },
    "delta_f1_pairing": {
        "fakerecogna_full":     {"prevalence": 0.026, "note": "DeltaF1~=-66% (media dos modelos)"},
        "fakerecogna_balanced": {"prevalence": 0.50,  "note": "DeltaF1~=-44% (media dos modelos)"},
        "interpretation": "mesma distancia de covariaveis, dois DeltaF1 -> label shift domina",
    },
}

fr_dist_df = pd.DataFrame(fakerecogna_distance["rows"]).set_index("comparacao")
print("Distancias FolhaUOL x FakeRecogna/economia.uol (metric-viz):")
display(fr_dist_df)

decomp = fakerecogna_distance["decomposition"]
decomp_df = pd.DataFrame({
    met: {
        "artefato_comprimento": decomp[met].get("length_artifact"),
        "deslocamento_dominio": decomp[met].get("domain_shift"),
        "total_vs_baseline":    decomp[met].get("total"),
        "pct_artefato_%":       decomp[met].get("pct_artifact"),
    }
    for met in ("EMD", "FTD2")
}).T
print("\nDecomposicao comprimento x dominio (EMD/FTD2):")
display(decomp_df)
print(f"MAUVE: queda de dominio = {decomp['MAUVE']['domain_shift_drop']} "
      f"({decomp['MAUVE']['note']})")
print("\n-> EMD/FTD2 entre FolhaUOL e FakeRecogna sao ~93-97% artefato de comprimento; "
      "use MAUVE como leitura de deslocamento de dominio.")

## 4. Descoberta dos modelos no Drive

Procura por `<RUNS_BASE>/<model_id>_<task>_test_set/model/` (convenção dos NBs 21 / 11 / 12 / 13). LLMs e ensembles ficam de fora — exigem orquestração extra.


In [ ]:
import re
import pandas as pd

RUN_PATTERN = re.compile(r"^(bert|tfidf)_.+_(binary|multiclass)_test_set$")

discovered = []
if not RUNS_BASE.exists():
    print(f"AVISO: RUNS_BASE nao existe: {RUNS_BASE}")
else:
    for run_dir in sorted(RUNS_BASE.iterdir()):
        if not run_dir.is_dir():
            continue
        m = RUN_PATTERN.match(run_dir.name)
        if not m:
            continue
        model_dir = run_dir / "model"
        if not model_dir.is_dir():
            # Sem pesos persistidos no Drive — pular silenciosamente
            continue
        try:
            model_type = fb.detect_model_type(model_dir)
        except FileNotFoundError as exc:
            print(f"  pula {run_dir.name}: {exc}")
            continue
        task = m.group(2)
        suffix = f"_{task}_test_set"
        model_id = run_dir.name[: run_dir.name.rfind(suffix)]
        discovered.append({
            "model_id": model_id,
            "task": task,
            "model_type": model_type,
            "model_dir": str(model_dir),
            "run_dir": str(run_dir),
        })

models_df = pd.DataFrame(discovered)
print(f"\n{len(models_df)} modelos com pesos no Drive:")
display(models_df[["model_id", "task", "model_type"]] if len(models_df) else models_df)
assert len(models_df) > 0, (
    f"Nenhum modelo encontrado em {RUNS_BASE}. Verifique se os NBs 21/11/12/13 "
    "salvaram os pesos no Drive (subdir 'model/' dentro de cada run)."
)



12 modelos com pesos no Drive:


,model_id,task,model_type
0,bert_bertimbau,binary,bert
1,bert_bertimbau,multiclass,bert
2,bert_deb3rta_base,binary,bert
3,bert_deb3rta_base,multiclass,bert
4,bert_finbert_ptbr,binary,bert
5,bert_finbert_ptbr,multiclass,bert
6,tfidf_linearsvc,binary,tfidf
7,tfidf_linearsvc,multiclass,tfidf
8,tfidf_logreg,binary,tfidf
9,tfidf_logreg,multiclass,tfidf


## 5. Avaliação em loop

Para cada modelo: 1 inferência por dataset (não 1 por nível). Os helpers `evaluate_level{1,2,3}_*` consomem o mesmo array `probs` para emitir cards independentes.


In [ ]:
import time

DEFAULT_BATCH = 64
DEFAULT_MAX_LEN = 128


def run_inference(model_dir, model_type, texts):
    if model_type == "bert":
        return fb.predict_bert(
            texts, model_dir,
            batch_size=DEFAULT_BATCH, max_length=DEFAULT_MAX_LEN,
        )
    return fb.predict_tfidf(texts, model_dir)


run_log = []
for _, row in models_df.iterrows():
    model_dir = Path(row["model_dir"])
    model_id = row["model_id"]
    task = row["task"]
    model_type = row["model_type"]
    print(f"\n=== {model_id} | task={task} | type={model_type} ===")

    model_size_mb = round(compute_artifact_size_mb(model_dir), 3)
    expected_classes = 2 if task == "binary" else 8
    t0 = time.perf_counter()
    log_row = {"model_id": model_id, "task": task}

    # --- Fake.Br ---
    try:
        fb_probs, fb_classes, fb_inf_s, fb_info = run_inference(model_dir, model_type, fb_texts)
    except Exception as exc:  # noqa: BLE001
        print(f"  ERRO Fake.Br inferencia: {type(exc).__name__}: {exc}")
        continue
    if fb_probs.shape[1] != expected_classes:
        print(f"  AVISO: modelo emite {fb_probs.shape[1]} classes; esperado {expected_classes}. Pulando.")
        continue
    print(f"  Fake.Br  inf: {fb_inf_s:.1f}s, shape={fb_probs.shape}")

    common_fb = dict(
        df=fb_df, probs=fb_probs, classes=fb_classes,
        model_id=model_id, model_type=model_type,
        output_root=OOD_RUNS_BASE, inference_seconds=fb_inf_s,
        model_size_mb=model_size_mb, max_length=DEFAULT_MAX_LEN,
        n_parameters=fb_info["n_parameters"], hardware=fb_info["hardware"],
    )
    if task == "binary":
        fb.evaluate_level1_binary(**common_fb)
        fb.evaluate_level3_subgroup(**common_fb)
    else:
        fb.evaluate_level2_multiclass(**common_fb)
    log_row["fb_inference_s"] = round(fb_inf_s, 2)

    # --- FakeRecogna ---
    try:
        fr_probs, fr_classes, fr_inf_s, fr_info = run_inference(model_dir, model_type, fr_texts)
    except Exception as exc:  # noqa: BLE001
        print(f"  ERRO FakeRecogna inferencia: {type(exc).__name__}: {exc}")
        continue
    print(f"  FakeReco inf: {fr_inf_s:.1f}s, shape={fr_probs.shape}")

    common_fr = dict(
        df=fr_df, probs=fr_probs, classes=fr_classes,
        model_id=model_id, model_type=model_type,
        output_root=OOD_RUNS_BASE, inference_seconds=fr_inf_s,
        model_size_mb=model_size_mb, max_length=DEFAULT_MAX_LEN,
        n_parameters=fr_info["n_parameters"], hardware=fr_info["hardware"],
    )
    if task == "binary":
        fr.evaluate_level1_binary(**common_fr)
        fr.evaluate_level3_uol_balanced(**common_fr, task="binary")
    else:
        fr.evaluate_level2_multiclass(**common_fr)
        fr.evaluate_level3_uol_balanced(**common_fr, task="multiclass")
    log_row["fr_inference_s"] = round(fr_inf_s, 2)

    # --- PortugueseNewsDataset + RecognaSumm (somente binario, label de topico direta) ---
    if task == "binary":
        try:
            pn_probs, pn_classes, pn_inf_s, pn_info = run_inference(model_dir, model_type, pn_texts)
        except Exception as exc:  # noqa: BLE001
            print(f"  ERRO PortugueseNews inferencia: {type(exc).__name__}: {exc}")
        else:
            print(f"  PortNews inf: {pn_inf_s:.1f}s, shape={pn_probs.shape}")
            pn.evaluate_level1_binary(
                df=pn_df, probs=pn_probs, classes=pn_classes,
                model_id=model_id, model_type=model_type,
                output_root=OOD_RUNS_BASE, inference_seconds=pn_inf_s,
                model_size_mb=model_size_mb, max_length=DEFAULT_MAX_LEN,
                n_parameters=pn_info["n_parameters"], hardware=pn_info["hardware"],
                partition=PN_PARTITION,
            )
            log_row["pn_inference_s"] = round(pn_inf_s, 2)

        try:
            rs_probs, rs_classes, rs_inf_s, rs_info = run_inference(model_dir, model_type, rs_texts)
        except Exception as exc:  # noqa: BLE001
            print(f"  ERRO RecognaSumm inferencia: {type(exc).__name__}: {exc}")
        else:
            print(f"  RecognaS inf: {rs_inf_s:.1f}s, shape={rs_probs.shape}")
            rs.evaluate_level1_binary(
                df=rs_df, probs=rs_probs, classes=rs_classes,
                model_id=model_id, model_type=model_type,
                output_root=OOD_RUNS_BASE, inference_seconds=rs_inf_s,
                model_size_mb=model_size_mb, max_length=DEFAULT_MAX_LEN,
                n_parameters=rs_info["n_parameters"], hardware=rs_info["hardware"],
                partition=RS_PARTITION,
            )
            log_row["rs_inference_s"] = round(rs_inf_s, 2)

    log_row["total_s"] = round(time.perf_counter() - t0, 2)
    run_log.append(log_row)

print("\n=== RUN LOG ===")
display(pd.DataFrame(run_log))

## 6. Agregação dos cards OOD

Walk em `OOD_RUNS_BASE/*/result_card.json` filtrando por `config.domain in {fake_br_full_texts, fake_recogna_economia_uol}`.


In [ ]:
import json

OOD_DOMAINS = {
    "fake_br_full_texts",
    "fake_recogna_economia_uol",
    "portuguese_news_wikinotices",
    "recognasumm_propor2024",
}

rows = []
for card_path in sorted(OOD_RUNS_BASE.glob("*/result_card.json")):
    card = json.loads(card_path.read_text())
    domain = card.get("config", {}).get("domain")
    if domain not in OOD_DOMAINS:
        continue
    metrics = card.get("metrics", {})
    rows.append({
        "model_id": card.get("model_id"),
        "task": card.get("task"),
        "domain": domain,
        "level": card.get("config", {}).get("level"),
        "partition": card.get("config", {}).get("partition"),
        "n_eval": card.get("n_eval_samples"),
        # Binarias (se aplicaveis)
        "f1": metrics.get("f1"),
        "precision": metrics.get("precision"),
        "recall": metrics.get("recall"),
        "auc_roc": metrics.get("auc_roc"),
        "brier": metrics.get("brier"),
        "ece": metrics.get("ece"),
        "positive_prevalence": metrics.get("positive_prevalence"),
        # Multiclasse
        "macro_f1": metrics.get("macro_f1"),
        "macro_f1_present_only": metrics.get("macro_f1_present_only"),
        "weighted_f1": metrics.get("weighted_f1"),
        "accuracy": metrics.get("accuracy"),
        "card_path": str(card_path.relative_to(OOD_RUNS_BASE)),
    })

cards_df = pd.DataFrame(rows).sort_values(["domain", "task", "level", "model_id"]).reset_index(drop=True)
print(f"{len(cards_df)} cards OOD agregados\n")
display(cards_df)

### 6.1 Pivot: F1 binário por (modelo × nível)

In [ ]:
if len(cards_df):
    pivot_bin = (
        cards_df[cards_df.task == "binary"]
        .pivot_table(index="model_id", columns=["domain", "level"], values="f1")
        .round(4)
    )
    print("F1 binario:")
    display(pivot_bin)

    pivot_macro = (
        cards_df[cards_df.task == "multiclass"]
        .pivot_table(index="model_id", columns=["domain", "level"], values="macro_f1")
        .round(4)
    )
    print("\nMacro-F1 multiclasse:")
    display(pivot_macro)


F1 binario:


domain              fake_br_full_texts                                  \
level             1_binary_full_corpus 3_subgroup_fake 3_subgroup_true   
model_id                                                                 
bert_bertimbau                  0.0983          0.2169          0.0608   
bert_deb3rta_base               0.1114          0.2095          0.0709   
bert_finbert_ptbr               0.1071          0.2143          0.0714   
tfidf_linearsvc                 0.1338          0.2817          0.0808   
tfidf_logreg                    0.1150          0.2128          0.0731   
tfidf_nb                        0.1462          0.2796          0.0865   

domain            fake_recogna_economia_uol                        
level                  1_binary_full_corpus 3_uol_balanced_binary  
model_id                                                           
bert_bertimbau                       0.3876                0.6247  
bert_deb3rta_base                    0.1713                0.3027  
bert_finbert_ptbr                    0.4122                0.6926  
tfidf_linearsvc                      0.1993                0.3077  
tfidf_logreg                         0.1785                0.3110  
tfidf_nb                             0.3508                0.5760


Macro-F1 multiclasse:


domain             fake_br_full_texts fake_recogna_economia_uol  \
level             2_multiclass_mapped       2_multiclass_mapped   
model_id                                                          
bert_bertimbau                 0.1406                    0.1916   
bert_deb3rta_base              0.1386                    0.1803   
bert_finbert_ptbr              0.1423                    0.2414   
tfidf_linearsvc                0.1688                    0.1866   
tfidf_logreg                   0.1625                    0.1906   
tfidf_nb                       0.2047                    0.2602   

domain                                       
level             3_uol_balanced_multiclass  
model_id                                     
bert_bertimbau                       0.1865  
bert_deb3rta_base                    0.0965  
bert_finbert_ptbr                    0.2054  
tfidf_linearsvc                      0.1648  
tfidf_logreg                         0.1601  
tfidf_nb                             0.2001

## 6.2 Distância × ΔF1 — deslocamento de covariáveis vs. label shift

Liga a magnitude do shift (§3.5–3.9) à **queda de desempenho** medida em §6. Para cada
modelo binário, `ΔF1 = F1_OOD − F1_teste_in-distribution` (carta `test_set` em
`<RUNS_BASE>`), e `ΔF1 relativa (%)` normaliza pela F1 in-distribution de cada modelo.

**Ponto central do `metric-viz` (reframe BRACIS).** Os positivos do FakeRecogna
(`economia.uol`) são **os mesmos** nos níveis *full* (prevalência ~2,6%) e *balanced*
(prevalência 50%) — logo têm **a mesma distância de covariáveis** $P(x)$. Mesmo assim o
ΔF1 difere muito entre eles (`metric-viz`: ≈ **−66%** full vs **−44%** balanced).
**Uma distância, dois ΔF1 ⟹ é o *label shift* (prevalência), não a distância de
covariáveis, que dirige aquele gap.** A tabela e o scatter abaixo recomputam esse
contraste a partir das cartas deste NB; os dois pontos do FakeRecogna caem na **mesma
vertical** (mesmo MMD²) com ΔF1 distintos.

In [ ]:
# F1 in-distribution (teste): carta test_set de cada modelo binario em RUNS_BASE
INDIST_F1 = {}
for run_dir in sorted(RUNS_BASE.glob("*_binary_test_set")):
    card_p = run_dir / "result_card.json"
    if not card_p.exists():
        continue
    c = json.loads(card_p.read_text())
    if c.get("task") == "binary" and c.get("regime") == "test_set":
        f1 = c.get("metrics", {}).get("f1")
        if f1 is not None:
            INDIST_F1[c["model_id"]] = float(f1)
print(f"F1 in-distribution (teste) de {len(INDIST_F1)} modelos binarios:")
for m, v in sorted(INDIST_F1.items()):
    print(f"  {m:<22s} {v:.4f}")

# DeltaF1 por (modelo, domain, level) nos niveis binarios do 6
delta_rows = []
bin_cards = cards_df[(cards_df.task == "binary") & cards_df.f1.notna()] if len(cards_df) else cards_df
for _, row in bin_cards.iterrows():
    base = INDIST_F1.get(row["model_id"])
    if not base:
        continue
    delta_rows.append({
        "model_id": row["model_id"], "domain": row["domain"], "level": row["level"],
        "f1_indist": round(base, 4), "f1_ood": round(row["f1"], 4),
        "delta_f1": round(row["f1"] - base, 4),
        "delta_f1_rel_%": round(100 * (row["f1"] - base) / base, 1),
        "prevalence": row["positive_prevalence"], "n_eval": row["n_eval"],
    })
delta_df = pd.DataFrame(delta_rows)

dom_to_ood = {
    "fake_br_full_texts": "fake_br",
    "fake_recogna_economia_uol": "fake_recogna",
    "portuguese_news_wikinotices": "portuguese_news_wikinotices",
    "recognasumm_propor2024": "recognasumm_propor2024",
}

if len(delta_df):
    delta_by_level = (
        delta_df.groupby(["domain", "level"])
        .agg(delta_f1_mean=("delta_f1", "mean"),
             delta_f1_rel_mean=("delta_f1_rel_%", "mean"),
             prevalence=("prevalence", "first"),
             n_models=("model_id", "nunique"))
        .round(3)
    )

    def _dist_lookup(domain, table, col):
        ood = dom_to_ood.get(domain)
        if ood is None or table is None or ood not in table.index or col not in table.columns:
            return np.nan
        return table.loc[ood, col]

    _mmd = shift_summary if "shift_summary" in globals() else None
    _dist = dist_summary if "dist_summary" in globals() else None
    dba = delta_by_level.reset_index()
    dba["mmd2"] = dba["domain"].map(lambda d: _dist_lookup(d, _mmd, "mmd2_mean"))
    dba["mauve"] = dba["domain"].map(lambda d: _dist_lookup(d, _dist, "MAUVE_mean"))
    print("\nDeltaF1 medio por (domain, level) x distancia:")
    display(dba.sort_values(["domain", "level"]).reset_index(drop=True))

    # contraste FakeRecogna full vs balanced (mesma distancia de covariaveis)
    k_full = ("fake_recogna_economia_uol", "1_binary_full_corpus")
    k_bal = ("fake_recogna_economia_uol", "3_uol_balanced_binary")
    if k_full in delta_by_level.index and k_bal in delta_by_level.index:
        ff, fb_ = delta_by_level.loc[k_full], delta_by_level.loc[k_bal]
        print("\n=== FakeRecogna: mesma distancia de covariaveis, dois DeltaF1 (label shift) ===")
        print(f"  full     (prev={ff['prevalence']}): DeltaF1 rel = {ff['delta_f1_rel_mean']:+.1f}%")
        print(f"  balanced (prev={fb_['prevalence']}): DeltaF1 rel = {fb_['delta_f1_rel_mean']:+.1f}%")
        print("  -> os positivos (economia.uol) sao os mesmos; so a prevalencia muda.")
        print("     A diferenca de DeltaF1 e label shift, nao distancia de P(x). "
              "(metric-viz: -66% full vs -44% balanced)")

    # scatter distancia (MMD2) x DeltaF1 relativo
    import matplotlib.pyplot as plt
    plot_df = dba.dropna(subset=["mmd2"]).copy()
    if len(plot_df):
        fig, ax = plt.subplots(figsize=(7.5, 5))
        ax.scatter(plot_df["mmd2"], plot_df["delta_f1_rel_mean"],
                   s=80, c="#2563eb", edgecolor="k", zorder=3)
        for _, r in plot_df.iterrows():
            lbl = f"{r['domain'].split('_')[0]}/{r['level'].split('_')[0]}"
            ax.annotate(lbl, (r["mmd2"], r["delta_f1_rel_mean"]),
                        textcoords="offset points", xytext=(6, 4), fontsize=8)
        ax.axhline(0, color="gray", lw=.8, ls=":")
        ax.set_xlabel("MMD^2 (3.5) -- deslocamento de P(x)")
        ax.set_ylabel("DeltaF1 relativo medio (%)")
        ax.set_title("Distancia de covariaveis x queda de F1 (OOD)\n"
                     "pontos FakeRecogna na mesma vertical = uma distancia, dois DeltaF1")
        plt.tight_layout(); plt.show()
else:
    print("AVISO: sem DeltaF1 (cards_df binario vazio ou cartas test_set ausentes em RUNS_BASE).")

## 7. McNemar pareado por (domain, level)

Apenas para o nível **binário** (McNemar é teste 2×2). Bonferroni aplicado automaticamente sobre `K*(K-1)/2` pares dentro de cada grupo.


In [ ]:
from collections import defaultdict

groups: dict[tuple, dict] = defaultdict(dict)
for card_path in sorted(OOD_RUNS_BASE.glob("*/result_card.json")):
    card = json.loads(card_path.read_text())
    if card.get("config", {}).get("domain") not in OOD_DOMAINS:
        continue
    if card.get("task") != "binary":
        continue
    pred_path = card_path.parent / "predictions.csv"
    if not pred_path.exists():
        continue
    pred_df = pd.read_csv(pred_path)
    if "index" not in pred_df.columns or "y_pred" not in pred_df.columns:
        continue
    key = (card["config"]["domain"], card["config"]["level"], card["task"])
    groups[key][card["model_id"]] = pred_df.set_index("index")

mcnemar_results: dict[tuple, pd.DataFrame] = {}
for (domain, level, task), preds_by_model in sorted(groups.items()):
    if len(preds_by_model) < 2:
        continue
    common_idx = None
    for df in preds_by_model.values():
        common_idx = df.index if common_idx is None else common_idx.intersection(df.index)
    aligned = {m: df.loc[common_idx, "y_pred"].to_numpy() for m, df in preds_by_model.items()}
    y_true = next(iter(preds_by_model.values())).loc[common_idx, "y_true"].to_numpy()

    pw = compute_mcnemar_pairwise(y_true, aligned)
    mcnemar_results[(domain, level, task)] = pw
    print(f"\n=== domain={domain} | level={level} | task={task} ===")
    print(f"  n={len(common_idx)}, k_modelos={len(aligned)}, n_pares={len(pw)}")
    display(pw[["method_a", "method_b", "p_value", "p_value_adjusted", "significant_after_correction"]])



=== domain=fake_br_full_texts | level=1_binary_full_corpus | task=binary ===
  n=7200, k_modelos=6, n_pares=15


,method_a,method_b,p_value,p_value_adjusted,significant_after_correction
0,bert_bertimbau,bert_deb3rta_base,0.617970,1.000000,False
1,bert_bertimbau,bert_finbert_ptbr,0.190430,1.000000,False
2,bert_bertimbau,tfidf_linearsvc,0.000000,0.000000,True
3,bert_bertimbau,tfidf_logreg,0.008151,0.122265,False
4,bert_bertimbau,tfidf_nb,0.000164,0.002463,True
5,bert_deb3rta_base,bert_finbert_ptbr,0.180194,1.000000,False
6,bert_deb3rta_base,tfidf_linearsvc,0.000000,0.000000,True
7,bert_deb3rta_base,tfidf_logreg,0.002073,0.031093,True
8,bert_deb3rta_base,tfidf_nb,0.000054,0.000816,True
9,bert_finbert_ptbr,tfidf_linearsvc,0.000000,0.000001,True



=== domain=fake_br_full_texts | level=3_subgroup_fake | task=binary ===
  n=3600, k_modelos=6, n_pares=15


,method_a,method_b,p_value,p_value_adjusted,significant_after_correction
0,bert_bertimbau,bert_deb3rta_base,0.026716,0.400735,False
1,bert_bertimbau,bert_finbert_ptbr,0.852684,1.000000,False
2,bert_bertimbau,tfidf_linearsvc,0.052204,0.783055,False
3,bert_bertimbau,tfidf_logreg,0.241317,1.000000,False
4,bert_bertimbau,tfidf_nb,0.799495,1.000000,False
5,bert_deb3rta_base,bert_finbert_ptbr,0.037813,0.567189,False
6,bert_deb3rta_base,tfidf_linearsvc,0.000048,0.000724,True
7,bert_deb3rta_base,tfidf_logreg,0.264288,1.000000,False
8,bert_deb3rta_base,tfidf_nb,0.059346,0.890197,False
9,bert_finbert_ptbr,tfidf_linearsvc,0.046945,0.704171,False



=== domain=fake_br_full_texts | level=3_subgroup_true | task=binary ===
  n=3600, k_modelos=6, n_pares=15


,method_a,method_b,p_value,p_value_adjusted,significant_after_correction
0,bert_bertimbau,bert_deb3rta_base,0.336515,1.000000,False
1,bert_bertimbau,bert_finbert_ptbr,0.079616,1.000000,False
2,bert_bertimbau,tfidf_linearsvc,0.000000,0.000000,True
3,bert_bertimbau,tfidf_logreg,0.000044,0.000660,True
4,bert_bertimbau,tfidf_nb,0.000004,0.000053,True
5,bert_deb3rta_base,bert_finbert_ptbr,0.862829,1.000000,False
6,bert_deb3rta_base,tfidf_linearsvc,0.000001,0.000019,True
7,bert_deb3rta_base,tfidf_logreg,0.002700,0.040497,True
8,bert_deb3rta_base,tfidf_nb,0.000328,0.004922,True
9,bert_finbert_ptbr,tfidf_linearsvc,0.000000,0.000003,True



=== domain=fake_recogna_economia_uol | level=1_binary_full_corpus | task=binary ===
  n=11872, k_modelos=6, n_pares=15


,method_a,method_b,p_value,p_value_adjusted,significant_after_correction
0,bert_bertimbau,bert_deb3rta_base,0.000070,0.001055,True
1,bert_bertimbau,bert_finbert_ptbr,0.088339,1.000000,False
2,bert_bertimbau,tfidf_linearsvc,0.607646,1.000000,False
3,bert_bertimbau,tfidf_logreg,0.000439,0.006585,True
4,bert_bertimbau,tfidf_nb,0.438578,1.000000,False
5,bert_deb3rta_base,bert_finbert_ptbr,0.008010,0.120149,False
6,bert_deb3rta_base,tfidf_linearsvc,0.000110,0.001644,True
7,bert_deb3rta_base,tfidf_logreg,0.698311,1.000000,False
8,bert_deb3rta_base,tfidf_nb,0.001117,0.016754,True
9,bert_finbert_ptbr,tfidf_linearsvc,0.476033,1.000000,False



=== domain=fake_recogna_economia_uol | level=3_uol_balanced_binary | task=binary ===
  n=624, k_modelos=6, n_pares=15


,method_a,method_b,p_value,p_value_adjusted,significant_after_correction
0,bert_bertimbau,bert_deb3rta_base,0.000000,0.000000,True
1,bert_bertimbau,bert_finbert_ptbr,0.001279,0.019185,True
2,bert_bertimbau,tfidf_linearsvc,0.000000,0.000000,True
3,bert_bertimbau,tfidf_logreg,0.000000,0.000000,True
4,bert_bertimbau,tfidf_nb,0.112924,1.000000,False
5,bert_deb3rta_base,bert_finbert_ptbr,0.000000,0.000000,True
6,bert_deb3rta_base,tfidf_linearsvc,0.725496,1.000000,False
7,bert_deb3rta_base,tfidf_logreg,0.905530,1.000000,False
8,bert_deb3rta_base,tfidf_nb,0.000000,0.000000,True
9,bert_finbert_ptbr,tfidf_linearsvc,0.000000,0.000000,True


## 8. Export para `<DRIVE>/reports/ood_evaluation/`

In [ ]:
REPORT_DIR = OOD_RUNS_BASE.parent / "reports" / "ood_evaluation"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

cards_df.to_csv(REPORT_DIR / "ood_cards_table.csv", index=False)
print(f"Tabela de cards: {REPORT_DIR / 'ood_cards_table.csv'}")

if len(cards_df):
    if (cards_df.task == "binary").any():
        cards_df[cards_df.task == "binary"].pivot_table(
            index="model_id", columns=["domain", "level"], values="f1",
        ).round(4).to_csv(REPORT_DIR / "ood_pivot_binary_f1.csv")
        print(f"Pivot F1 binario: {REPORT_DIR / 'ood_pivot_binary_f1.csv'}")
    if (cards_df.task == "multiclass").any():
        cards_df[cards_df.task == "multiclass"].pivot_table(
            index="model_id", columns=["domain", "level"], values="macro_f1",
        ).round(4).to_csv(REPORT_DIR / "ood_pivot_multiclass_macrof1.csv")
        print(f"Pivot macro-F1 multi: {REPORT_DIR / 'ood_pivot_multiclass_macrof1.csv'}")

for (domain, level, task), pw in mcnemar_results.items():
    fname = f"mcnemar_{domain}_{level}_{task}.csv"
    pw.to_csv(REPORT_DIR / fname, index=False)
    print(f"McNemar: {REPORT_DIR / fname}")

# Shift representacional (3.5-3.7). Tolerante a execucao parcial do NB.
try:
    shift_summary.to_csv(REPORT_DIR / "shift_mmd2_kts_summary.csv")
    shift_df.to_csv(REPORT_DIR / "shift_mmd2_kts_runs.csv", index=False)
    print(f"Shift MMD2/KTS (summary): {REPORT_DIR / 'shift_mmd2_kts_summary.csv'}")
    print(f"Shift MMD2/KTS (runs)   : {REPORT_DIR / 'shift_mmd2_kts_runs.csv'}")
except NameError:
    print("AVISO: shift_summary/shift_df nao calculados (3.5-3.7 nao rodou); pulando export.")

# Distancias EMD/FTD2/MAUVE (3.8). Tolerante a execucao parcial.
try:
    dist_summary.to_csv(REPORT_DIR / "ood_distances_emd_ftd_mauve.csv")
    dist_df.to_csv(REPORT_DIR / "ood_distances_runs.csv", index=False)
    print(f"Distancias EMD/FTD2/MAUVE: {REPORT_DIR / 'ood_distances_emd_ftd_mauve.csv'}")
except NameError:
    print("AVISO: dist_summary/dist_df nao calculados (3.8 nao rodou); pulando export.")

# Decomposicao comprimento x dominio do FakeRecogna (3.9, metric-viz).
try:
    fr_dist_df.to_csv(REPORT_DIR / "fakerecogna_distance_metricviz.csv")
    (REPORT_DIR / "fakerecogna_distance_metricviz.json").write_text(
        json.dumps(fakerecogna_distance, ensure_ascii=False, indent=2))
    print(f"FakeRecogna distancia (metric-viz): {REPORT_DIR / 'fakerecogna_distance_metricviz.csv'}")
except NameError:
    print("AVISO: fr_dist_df nao definido (3.9 nao rodou); pulando export.")

# Distancia x DeltaF1 (6.2).
try:
    delta_df.to_csv(REPORT_DIR / "distance_vs_deltaf1_per_model.csv", index=False)
    dba.to_csv(REPORT_DIR / "distance_vs_deltaf1_by_level.csv", index=False)
    print(f"Distancia x DeltaF1: {REPORT_DIR / 'distance_vs_deltaf1_by_level.csv'}")
except NameError:
    print("AVISO: delta_df/dba nao definidos (6.2 nao rodou); pulando export.")

print(f"\nDestino: {REPORT_DIR}")